# Dispatch Visualization Draft

This notebook rebuilds the optimizer decision variables from the saved daily inputs and renders an interactive dispatch dashboard.

In [1]:
from datetime import date
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
PROJECT_ROOT = next(
    candidate for candidate in candidates if (candidate / "energy_planner" / "src").exists()
)
SRC_ROOT = PROJECT_ROOT / "energy_planner" / "src"

for candidate in (PROJECT_ROOT, SRC_ROOT):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

import pandas as pd
from IPython.display import Markdown, display

from ingestion.load_predicted_inputs import load_predicted_inputs
from main import _run_optimizer
from reporting.optimization_summary import (
    build_optimization_summary_payload,
    try_generate_llm_summary,
)
from state.load_state import load_current_state
from visualization import (
    build_visualization_frame,
    create_dispatch_dashboard,
    save_dashboard_html,
    summarize_dispatch,
)


In [2]:
run_date = date(2026, 3, 11)
llm_base_url = "https://openrouter.ai/api/v1"
llm_model = "openrouter/free"
llm_api_key = "TO SET"


if not llm_api_key or llm_api_key == "PUT_YOUR_OPENROUTER_API_KEY_HERE":
    raise RuntimeError(
        "Missing API key. Replace the placeholder with your own OpenRouter key before running this notebook."
    )


predicted_inputs = load_predicted_inputs(run_date=run_date)
state = load_current_state(run_date=run_date)
plan_df = _run_optimizer(predicted_inputs, state)

if plan_df is None:
    raise RuntimeError("Optimizer plan could not be created. Check the CPLEX installation.")

viz_df = build_visualization_frame(
    predicted_inputs,
    plan_df,
    initial_battery_kwh=state["E_bat_0"],
    battery_capacity_kwh=state["E_max"],
)

summary = summarize_dispatch(viz_df)
summary_payload = build_optimization_summary_payload(
    predicted_inputs,
    plan_df,
    initial_battery_kwh=state["E_bat_0"],
    battery_capacity_kwh=state["E_max"],
)
summary_text, summary_source = try_generate_llm_summary(
    summary_payload,
    model=llm_model,
    api_key=llm_api_key,
    base_url=llm_base_url,
)
summary_markdown = "## Resume naturel du dispatch\n\n" \
    + f"Source: `{summary_source}` | Modele: `{llm_model}`\n\n" \
    + summary_text
summary


Statut de la solution : 101 - integer optimal solution
Valeur de la fonction objectif : -5.875691400000005

Heure | Pin  | Pgo  | PV   | Pch | Pdis | Ebat | S
    0 | 4.19 | 0.00 | 0.00 | 4.00 | 0.00 | 10.00 | 1
    1 | 3.71 | 0.00 | 0.00 | 3.50 | 0.00 | 13.50 | 1
    2 | 0.14 | 0.00 | 0.00 | 0.00 | 0.00 | 13.50 | 1
    3 | 0.20 | 0.00 | 0.00 | 0.00 | 0.00 | 13.50 | 1
    4 | 0.38 | 0.00 | 0.00 | 0.00 | 0.00 | 13.50 | 1
    5 | 0.93 | 0.00 | 0.00 | 0.00 | 0.00 | 13.50 | 1
    6 | 1.37 | 0.00 | 0.00 | 0.00 | 0.00 | 13.50 | 1
    7 | 1.55 | 0.00 | 0.00 | 0.00 | 0.48 | 13.02 | 1
    8 | 0.00 | 0.00 | 0.00 | 0.00 | 2.61 | 10.41 | 1
    9 | 0.00 | 0.00 | 0.00 | 0.00 | 2.41 | 8.00 | 1
   10 | 0.00 | 0.00 | 0.00 | 0.00 | 2.26 | 5.74 | 1
   11 | 0.00 | 0.00 | 0.00 | 0.00 | 2.36 | 3.38 | 1
   12 | 0.00 | 0.00 | 0.00 | 0.00 | 1.88 | 1.50 | 1
   13 | 5.76 | 0.00 | 0.00 | 4.00 | 0.00 | 5.50 | 1
   14 | 5.71 | 0.00 | 0.00 | 4.00 | 0.00 | 9.50 | 1
   15 | 5.25 | 0.00 | 0.00 | 4.00 | 0.00 | 13.50 | 1

{'total_served_demand_kWh': 39.549,
 'total_pv_kWh': 0.008,
 'grid_import_kWh': 33.541,
 'grid_export_kWh': 0.0,
 'battery_charge_kWh': 19.5,
 'battery_discharge_kWh': 25.5,
 'peak_battery_kWh': 13.5,
 'regime_hours': {'Battery support': 11, 'Grid charge': 5, 'Grid support': 8}}

In [3]:
viz_df[
    [
        "hour",
        "Pfix_pred_kW",
        "Pflex_pred_kW",
        "total_demand_kW",
        "PV",
        "Pin",
        "Pgo",
        "Pch",
        "Pdis",
        "Ebat",
        "regime",
    ]
]


,hour,Pfix_pred_kW,Pflex_pred_kW,total_demand_kW,PV,Pin,Pgo,Pch,Pdis,Ebat,regime
0,0,0.190,0.003,0.193,0.000,4.193,0.0,4.0,0.000,10.000,Grid charge
1,1,0.129,0.086,0.215,0.000,3.715,0.0,3.5,0.000,13.500,Grid charge
2,2,0.136,0.003,0.139,0.000,0.139,0.0,0.0,0.000,13.500,Grid support
3,3,0.195,0.003,0.198,0.000,0.198,0.0,0.0,0.000,13.500,Grid support
4,4,0.294,0.087,0.381,0.000,0.381,0.0,0.0,0.000,13.500,Grid support
5,5,0.327,0.603,0.930,0.000,0.930,0.0,0.0,0.000,13.500,Grid support
6,6,0.231,1.143,1.374,0.000,1.374,0.0,0.0,0.000,13.500,Grid support
7,7,0.122,1.904,2.026,0.000,1.546,0.0,0.0,0.480,13.020,Grid support
8,8,0.894,1.720,2.614,0.000,0.000,0.0,0.0,2.614,10.406,Battery support
9,9,0.827,1.582,2.409,0.000,0.000,0.0,0.0,2.409,7.997,Battery support


In [4]:
fig = create_dispatch_dashboard(
    viz_df,
    # title=f"Smart Building Dispatch Dashboard | {run_date.isoformat()}",
)
fig.show()


In [5]:
output_path = PROJECT_ROOT / "energy_planner" / "data" / "processed" / f"dispatch_dashboard_{run_date.isoformat()}.html"
saved = save_dashboard_html(fig, output_path)
saved


WindowsPath('C:/Users/barra/Desktop/Cours 3A CentraleSupelec/Projet Dassault/Projet-Dassault-Smart-Building/energy_planner/data/processed/dispatch_dashboard_2026-03-11.html')

In [6]:
display(Markdown(summary_markdown))


## Resume naturel du dispatch

Source: `template_llm_error` | Modele: `openrouter/free`

Resume optimisation sur 24h : le batiment sert 39.55 kWh de demande. Seuls 33.54 kWh sont achetes au reseau, soit 84.8% de la demande servie (cout 6.25 EUR). Le batiment revend 0.00 kWh (revenu 0.00 EUR), pour un cout net de 6.25 EUR. La batterie charge 19.50 kWh, decharge 25.50 kWh et passe de 6.00 a 0.00 kWh (pic a 13.50 kWh). La flexibilite servie represente 100.0% de la demande flexible. Le pic d'achat reseau est atteint a 13h avec 5.76 kW, et le pic de vente a 00h avec 0.00 kW. Le regime dominant est 'Battery support' sur 11 heures.